In [0]:
%run "/Workspace/Users/malhotrasushant32@gmail.com/(Clone) Basic Checks"

In [0]:
df_list=[]
mount_path = "/mnt/gsynergyproject/rawdata"
file_list = ["fact.averagecosts.dlm",
"fact.transactions.dlm",
"hier.clnd.dlm",
"hier.hldy.dlm",
"hier.invloc.dlm",
"hier.invstatus.dlm",
"hier.possite.dlm",
"hier.pricestate.dlm",
"hier.prod.dlm",
"hier.rtlloc.dlm"]
read_files(mount_path, file_list)

In [0]:
def read_files(mount_path, file_list):
    for file_name in file_list:
        new_name = file_name.replace(".", "_").replace("dlm", "df") 
        file_path = f"{mount_path}/{file_name}" 

        
        
        
        df = spark.read.option("header", "true") \
                       .option("delimiter", "|") \
                       .option("inferSchema", "true") \
                       .csv(file_path)
        
        
        globals()[new_name] = df  
        
        df_list.append(new_name)

In [0]:
print(df_list)

['fact_averagecosts_df', 'fact_transactions_df', 'hier_clnd_df', 'hier_hldy_df', 'hier_invloc_df', 'hier_invstatus_df', 'hier_possite_df', 'hier_pricestate_df', 'hier_prod_df', 'hier_rtlloc_df']


In [0]:
expected_schemas = {
    "fact_averagecosts_df": {
        "fscldt_id": "int",  # Primary Key, FK to hier_clnd_df
        "sku_id": "string",  # Primary Key, FK to hier_prod_df
        "average_unit_standardcost": "double",
        "average_unit_landedcost": "double",
    },
    "fact_transactions_df": {
        "order_id": "bigint",  # Primary Key
        "line_id": "int",  # Primary Key
        "type": "string",
        "dt": "timestamp",
        "pos_site_id": "string",  # FK to hier_possite_df
        "sku_id": "string",  # FK to hier_prod_df
        "fscldt_id": "int",  # FK to hier_clnd_df
        "price_substate_id": "string",  # FK to hier_pricestate_df
        "sales_units": "int",
        "sales_dollars": "double",
        "discount_dollars": "double",
        "original_order_id": "bigint",
        "original_line_id": "int",
    },
    "hier_clnd_df": {
        "fscldt_id": "int",  # Primary Key
        "fscldt_label": "string",
        "fsclwk_id": "int",
        "fsclwk_label": "string",
        "fsclmth_id": "int",
        "fsclmth_label": "string",
        "fsclqrtr_id": "int",
        "fsclqrtr_label": "string",
        "fsclyr_id": "int",
        "fsclyr_label": "string",
        "ssn_id": "string",
        "ssn_label": "string",
    },
    "hier_hldy_df": {
        "hldy_id": "string",  # Primary Key
        "hldy_label": "string",
    },
    "hier_invloc_df": {
        "loc": "int",  # Primary Key
        "loc_label": "string",
        "loctype": "string",
        "loctype_label": "string",
    },
    "hier_invstatus_df": {
        "code_id": "string",  # Primary Key
        "code_label": "string",
        "bckt_id": "string",
        "bckt_label": "string",
        "ownrshp_id": "string",
        "ownrshp_label": "string",
    },
    "hier_possite_df": {
        "site_id": "string",  # Primary Key
        "site_label": "string",
        "subchnl_id": "string",
        "subchnl_label": "string",
        "chnl_id": "string",
        "chnl_label": "string",
    },
    "hier_pricestate_df": {
        "substate_id": "string",  # Primary Key
        "substate_label": "string",
        "state_id": "string",
        "state_label": "string",
    },
    "hier_prod_df": {
        "sku_id": "string",  # Primary Key
        "sku_label": "string",
        "stylclr_id": "string",
        "stylclr_label": "string",
        "styl_id": "string",
        "styl_label": "string",
        "subcat_id": "int",
        "subcat_label": "string",
        "cat_id": "int",
        "cat_label": "string",
        "dept_id": "int",
        "dept_label": "string",
        "issvc": "int",
        "isasmbly": "int",
        "isnfs": "int",
    },
    "hier_rtlloc_df": {
        "str": "int",  # Primary Key
        "str_label": "string",
        "dstr": "int",
        "dstr_label": "string",
        "rgn": "int",
        "rgn_label": "string",
    },
}

primary_keys = {
    "fact_averagecosts_df": ["fscldt_id", "sku_id"],
    "fact_transactions_df": ["order_id", "line_id"],
    "hier_clnd_df": "fscldt_id",
    "hier_hldy_df": "hldy_id",
    "hier_invloc_df": "loc",
    "hier_invstatus_df": "code_id",
    "hier_possite_df": "site_id",
    "hier_pricestate_df": "substate_id",
    "hier_prod_df": "sku_id",
    "hier_rtlloc_df": "str",
}

foreign_keys = [
    ("fact_transactions_df", "hier_possite_df", "pos_site_id", "site_id"),
    ("fact_transactions_df", "hier_prod_df", "sku_id", "sku_id"),
    ("fact_transactions_df", "hier_clnd_df", "fscldt_id", "fscldt_id"),
    ("fact_transactions_df", "hier_pricestate_df", "price_substate_id", "substate_id"),
    ("fact_averagecosts_df", "hier_clnd_df", "fscldt_id", "fscldt_id"),
    ("fact_averagecosts_df", "hier_prod_df", "sku_id", "sku_id"),
]



In [0]:
for df_name in df_list:  
    df = globals().get(df_name)  
    
    if df is not None: 
        cols = df.columns
        check_null_values(df, df_name)
        check_primary_key_uniqueness(df, df_name, primary_keys[df_name])
        check_data_types(df, df_name, expected_schemas[df_name])  
    else:
        print(f"Warning: DataFrame '{df_name}' not found.")


Checking null values for DataFrame: fact_averagecosts_df
Null Value check for DataFrame 'fact_averagecosts_df' is completed!

hello
Checking uniqueness for primary key ['fscldt_id', 'sku_id'] in DataFrame: fact_averagecosts_df
Primary key ['fscldt_id', 'sku_id'] is unique in 'fact_averagecosts_df'.
Primary Key Uniqueness check for 'fact_averagecosts_df' is completed!

Checking data types for DataFrame: fact_averagecosts_df
{'fscldt_id': 'int', 'sku_id': 'string', 'average_unit_standardcost': 'double', 'average_unit_landedcost': 'double'}
Column 'fscldt_id' in 'fact_averagecosts_df' has the correct data type: int.
Column 'sku_id' in 'fact_averagecosts_df' has the correct data type: string.
Column 'average_unit_standardcost' in 'fact_averagecosts_df' has the correct data type: double.
Column 'average_unit_landedcost' in 'fact_averagecosts_df' has the correct data type: double.
Data Type check for 'fact_averagecosts_df' is completed!

Checking null values for DataFrame: fact_transactions_

In [0]:
df = globals().get("hier_clnd_df")

check_primary_key_uniqueness(df, "hier_clnd_df", "fscldt_id")

hello
Checking uniqueness for primary key fscldt_id in DataFrame: hier_clnd_df
Primary key fscldt_id is unique in 'hier_clnd_df'.
Primary Key Uniqueness check for 'hier_clnd_df' is completed!



In [0]:
for fact_df_name, dim_df_name, fact_fk, dim_pk in foreign_keys:
    fact_df = globals().get(fact_df_name)
    dim_df = globals().get(dim_df_name)
    
    if fact_df is not None and dim_df is not None:
        check_foreign_key_constraint(fact_df, fact_df_name, dim_df, dim_df_name, fact_fk, dim_pk)
    else:
        print(f" Warning: One of the DataFrames '{fact_df_name}' or '{dim_df_name}' is missing.")


🔍 Checking Foreign Key Constraint between 'fact_transactions_df' (fact) and 'hier_possite_df' (dimension) on 'pos_site_id' → 'site_id'
Foreign Key Constraint 'pos_site_id → site_id' is satisfied.
 Foreign Key Constraint check for 'fact_transactions_df' → 'hier_possite_df' is completed!

🔍 Checking Foreign Key Constraint between 'fact_transactions_df' (fact) and 'hier_prod_df' (dimension) on 'sku_id' → 'sku_id'
Foreign Key Constraint 'sku_id → sku_id' is satisfied.
 Foreign Key Constraint check for 'fact_transactions_df' → 'hier_prod_df' is completed!

🔍 Checking Foreign Key Constraint between 'fact_transactions_df' (fact) and 'hier_clnd_df' (dimension) on 'fscldt_id' → 'fscldt_id'
2206380 records in 'fact_transactions_df' have missing foreign key values that do not exist in 'hier_clnd_df'.
 Foreign Key Constraint check for 'fact_transactions_df' → 'hier_clnd_df' is completed!

🔍 Checking Foreign Key Constraint between 'fact_transactions_df' (fact) and 'hier_pricestate_df' (dimension) o

TASK 2


In [0]:
stg_department_df = hier_prod_df.select("dept_id", "dept_label").distinct()
stg_category_df = hier_prod_df.select("cat_id", "cat_label", "dept_id").distinct()
stg_subcategory_df = hier_prod_df.select("subcat_id", "subcat_label", "cat_id").distinct()
stg_style_df = hier_prod_df.select("styl_id", "styl_label", "subcat_id").distinct()
stg_style_color_df = hier_prod_df.select("stylclr_id", "stylclr_label", "styl_id").distinct()
stg_sku_df = hier_prod_df.select("sku_id", "sku_label", "stylclr_id", "issvc", "isasmbly", "isnfs").distinct()

#  Normalizing Store Hierarchy
stg_region_df = hier_rtlloc_df.select("rgn", "rgn_label").distinct().withColumnRenamed("rgn", "rgn_id")
stg_district_df = hier_rtlloc_df.select("dstr", "dstr_label", "rgn").distinct().withColumnRenamed("dstr", "dstr_id").withColumnRenamed("rgn", "rgn_id")
stg_store_df = hier_rtlloc_df.select("str", "str_label", "dstr").distinct().withColumnRenamed("str", "str_id").withColumnRenamed("dstr", "dstr_id")

# Normalizing Time Hierarchy
stg_fiscal_calendar_df = hier_clnd_df.select("fscldt_id", "fscldt_label", "fsclwk_id", "fsclmth_id", "fsclqrtr_id", "fsclyr_id", "ssn_id").distinct()
stg_holidays_df = hier_hldy_df.select("hldy_id", "hldy_label").distinct()

#  Normalizing POS Hierarchy
stg_channel_df = hier_possite_df.select("chnl_id", "chnl_label").distinct()
stg_subchannel_df = hier_possite_df.select("subchnl_id", "subchnl_label", "chnl_id").distinct()
stg_pos_site_df = hier_possite_df.select("site_id", "site_label", "subchnl_id").distinct()

#  Staging Fact Tables (with normalized FK references)
stg_fact_transactions_df = (
    fact_transactions_df
    .select(
        "order_id", "line_id", "type", "dt", "pos_site_id", "sku_id", "fscldt_id",
        "price_substate_id", "sales_units", "sales_dollars", "discount_dollars",
        "original_order_id", "original_line_id"
    )
)

stg_fact_averagecosts_df = fact_averagecosts_df.select("fscldt_id", "sku_id", "average_unit_standardcost", "average_unit_landedcost")

# Save Processed Data to a Staging Location (Database or Parquet)
stg_department_df.write.mode("overwrite").saveAsTable("stg_department")
stg_category_df.write.mode("overwrite").saveAsTable("stg_category")
stg_subcategory_df.write.mode("overwrite").saveAsTable("stg_subcategory")
stg_style_df.write.mode("overwrite").saveAsTable("stg_style")
stg_style_color_df.write.mode("overwrite").saveAsTable("stg_style_color")
stg_sku_df.write.mode("overwrite").saveAsTable("stg_sku")

stg_region_df.write.mode("overwrite").saveAsTable("stg_region")
stg_district_df.write.mode("overwrite").saveAsTable("stg_district")
stg_store_df.write.mode("overwrite").saveAsTable("stg_store")

stg_fiscal_calendar_df.write.mode("overwrite").saveAsTable("stg_fiscal_calendar")
stg_holidays_df.write.mode("overwrite").saveAsTable("stg_holidays")

stg_channel_df.write.mode("overwrite").saveAsTable("stg_channel")
stg_subchannel_df.write.mode("overwrite").saveAsTable("stg_subchannel")
stg_pos_site_df.write.mode("overwrite").saveAsTable("stg_pos_site")

stg_fact_transactions_df.write.mode("overwrite").saveAsTable("stg_fact_transactions")
stg_fact_averagecosts_df.write.mode("overwrite").saveAsTable("stg_fact_averagecosts")

print("Data has been normalized and saved to staging tables!")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:728)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:446)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:446)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
foreign_keys_staging = [
    # Product Hierarchy
    ("stg_sku_df", "stg_style_color_df", "stylclr_id", "stylclr_id"),
    ("stg_style_color_df", "stg_style_df", "styl_id", "styl_id"),
    ("stg_style_df", "stg_subcategory_df", "subcat_id", "subcat_id"),
    ("stg_subcategory_df", "stg_category_df", "cat_id", "cat_id"),
    ("stg_category_df", "stg_department_df", "dept_id", "dept_id"),

    # Store Hierarchy
    ("stg_store_df", "stg_district_df", "dstr_id", "dstr_id"),
    ("stg_district_df", "stg_region_df", "rgn_id", "rgn_id"),

   
    # POS Hierarchy
    ("stg_pos_site_df", "stg_subchannel_df", "subchnl_id", "subchnl_id"),
    ("stg_subchannel_df", "stg_channel_df", "chnl_id", "chnl_id"),

    # Fact Table Foreign Key References
    ("stg_fact_transactions_df", "stg_sku_df", "sku_id", "sku_id"),
    ("stg_fact_transactions_df", "stg_pos_site_df", "pos_site_id", "site_id"),
    ("stg_fact_transactions_df", "stg_fiscal_calendar_df", "fscldt_id", "fscldt_id"),

    ("stg_fact_averagecosts_df", "stg_sku_df", "sku_id", "sku_id"),
    ("stg_fact_averagecosts_df", "stg_fiscal_calendar_df", "fscldt_id", "fscldt_id"),
]


# Run Foreign Key Check
for fact_df_name, dim_df_name, fact_fk, dim_pk in foreign_keys_staging:
    fact_df = globals().get(fact_df_name)
    dim_df = globals().get(dim_df_name)
    
    if fact_df is not None and dim_df is not None:
        check_foreign_key_constraint(fact_df, fact_df_name, dim_df, dim_df_name, fact_fk, dim_pk)
    else:
        print(f"Warning: One of the DataFrames '{fact_df_name}' or '{dim_df_name}' is missing.")


🔍 Checking Foreign Key Constraint between 'stg_sku_df' (fact) and 'stg_style_color_df' (dimension) on 'stylclr_id' → 'stylclr_id'
Foreign Key Constraint 'stylclr_id → stylclr_id' is satisfied.
 Foreign Key Constraint check for 'stg_sku_df' → 'stg_style_color_df' is completed!

🔍 Checking Foreign Key Constraint between 'stg_style_color_df' (fact) and 'stg_style_df' (dimension) on 'styl_id' → 'styl_id'
Foreign Key Constraint 'styl_id → styl_id' is satisfied.
 Foreign Key Constraint check for 'stg_style_color_df' → 'stg_style_df' is completed!

🔍 Checking Foreign Key Constraint between 'stg_style_df' (fact) and 'stg_subcategory_df' (dimension) on 'subcat_id' → 'subcat_id'
Foreign Key Constraint 'subcat_id → subcat_id' is satisfied.
 Foreign Key Constraint check for 'stg_style_df' → 'stg_subcategory_df' is completed!

🔍 Checking Foreign Key Constraint between 'stg_subcategory_df' (fact) and 'stg_category_df' (dimension) on 'cat_id' → 'cat_id'
Foreign Key Constraint 'cat_id → cat_id' is sat

TASK 3

In [0]:
from pyspark.sql.functions import sum

mview_weekly_sales_df = (
    fact_transactions_df.join(hier_clnd_df.select("fscldt_id", "fsclwk_id"), on="fscldt_id", how="left")
    .groupBy("pos_site_id", "sku_id", "fsclwk_id", "price_substate_id", "type")
    .agg(
        sum("sales_units").alias("total_sales_units"),
        sum("sales_dollars").alias("total_sales_dollars"),
        sum("discount_dollars").alias("total_discount_dollars"),
    )
)



In [0]:
dbutils.fs.rm("/mnt/gsynergyproject/processeddata/", True)
mview_weekly_sales_df.write.format("delta").mode("overwrite").save("/mnt/gsynergyproject/processeddata/mview_weekly_sales_table")

In [0]:
%sql
SELECT * FROM delta.`/mnt/gsynergyproject/processeddata/mview_weekly_sales_table`


pos_site_id,sku_id,fsclwk_id,price_substate_id,type,total_sales_units,total_sales_dollars,total_discount_dollars
CATMAIN,2785140701,201801,FP,Sale,16,1108.71,10.49
177,1AV5420000,201801,MD2,Sale,9,49.89,39.93000000000001
155,2598420801,201801,FP,Sale,1,59.95,0.0
INETMAIN,0310920000,201801,FP,Return,3,126.0,0.0
CATMAIN,6831981800,201801,FP,Sale,3,97.86000000000001,6.99
INETMAIN,2AL8220601,201801,FP,Sale,8,639.6,0.0
CATSALE,2666820701,201801,FP,Return,5,197.75,2.0
CATSALE,2AB1530801,201801,MD1,Return,3,149.97,0.0
INETMAIN,0787630000,201801,FP,Sale,1,12.0,0.0
171,0174410000,201801,FP,Sale,3,86.25,0.0


In [0]:
from pyspark.sql.functions import sum
from delta.tables import DeltaTable


latest_fact_df = fact_transactions_df.filter("dt >= date_sub(current_date(),10 )") 

latest_fact_with_week_df = latest_fact_df.join(
    hier_clnd_df.select("fscldt_id", "fsclwk_id"),
    on="fscldt_id",
    how="left"
)

incremental_aggregates_df = (
    latest_fact_with_week_df
    .groupBy("pos_site_id", "sku_id", "fsclwk_id", "price_substate_id", "type")
    .agg(
        sum("sales_units").alias("total_sales_units"),
        sum("sales_dollars").alias("total_sales_dollars"),
        sum("discount_dollars").alias("total_discount_dollars")
    )
)

# Step 4: Merge into existing mview_weekly_sales Delta Table
mview_table_path = "/mnt/gsynergyproject/processeddata/mview_weekly_sales_table"

mview_table = DeltaTable.forPath(spark, mview_table_path)

mview_table.alias("existing").merge(
    incremental_aggregates_df.alias("updates"),
    """
    existing.pos_site_id = updates.pos_site_id
    AND existing.sku_id = updates.sku_id
    AND existing.fsclwk_id = updates.fsclwk_id
    AND existing.price_substate_id = updates.price_substate_id
    AND existing.type = updates.type
    """
).whenMatchedUpdate(set={
    "total_sales_units": "existing.total_sales_units + updates.total_sales_units",
    "total_sales_dollars": "existing.total_sales_dollars + updates.total_sales_dollars",
    "total_discount_dollars": "existing.total_discount_dollars + updates.total_discount_dollars"
}).whenNotMatchedInsert(values={
    "pos_site_id": "updates.pos_site_id",
    "sku_id": "updates.sku_id",
    "fsclwk_id": "updates.fsclwk_id",
    "price_substate_id": "updates.price_substate_id",
    "type": "updates.type",
    "total_sales_units": "updates.total_sales_units",
    "total_sales_dollars": "updates.total_sales_dollars",
    "total_discount_dollars": "updates.total_discount_dollars"
}).execute()


In [0]:
%sql
select * from delta.`/mnt/gsynergyproject/processeddata/mview_weekly_sales_table` w
join delta.`/mnt/gsynergyproject/processeddata/hier_clnd_table` c on w.fsclwk_id = c.fsclwk_id
where c.date=date_sub(current_date(), 7)


pos_site_id,sku_id,fsclwk_id,price_substate_id,type,total_sales_units,total_sales_dollars,total_discount_dollars,fscldt_id,fscldt_label,fsclwk_id,fsclwk_label,fsclmth_id,fsclmth_label,fsclqrtr_id,fsclqrtr_label,fsclyr_id,fsclyr_label,ssn_id,ssn_label,ly_fscldt_id,lly_fscldt_id,fscldow,fscldom,fscldoq,fscldoy,fsclwoy,fsclmoy,fsclqoy,date


In [0]:
latest_fact_with_week_df.write.format("delta").mode("overwrite").save("/mnt/gsynergyproject/processeddata/latest_fact_with_week_df")


In [0]:
%sql
select * from delta.`/mnt/gsynergyproject/processeddata/latest_fact_with_week_df`

fscldt_id,order_id,line_id,type,dt,pos_site_id,sku_id,price_substate_id,sales_units,sales_dollars,discount_dollars,original_order_id,original_line_id,fsclwk_id
20001412,164099145,3,bikna,2025-03-27T17:05:33Z,Sushant,123456,malh,2,5.95,0.0,null,null,null
2010203,164099145,3,Sale,2025-03-27T17:05:33Z,CATM,26689601,F,2,5.95,0.0,null,null,null


In [0]:
latest_fact_with_week_dffrom pyspark.sql import Row
from datetime import datetime
new_row = [
    Row(
        order_id=164099145, line_id=3, type="bikna", 
        dt=datetime.strptime("2025-03-27T17:05:33Z", "%Y-%m-%dT%H:%M:%SZ"),  # Convert here
        pos_site_id="Sushant", sku_id=123456, fscldt_id=20001412, 
        price_substate_id="malh", sales_units=2, sales_dollars=5.95, 
        discount_dollars=0.0, original_order_id=None, original_line_id=None
    )
]
# Create a new DataFrame (ensure schema matches `fact_transactions_df`)
new_row_df = spark.createDataFrame(new_row, schema=fact_transactions_df.schema)

fact_transactions_df=fact_transactions_df.unionByName(new_row_df)


In [0]:
display(fact_transactions_df.filter(col("pos_site_id")=="Sushant"))

order_id,line_id,type,dt,pos_site_id,sku_id,fscldt_id,price_substate_id,sales_units,sales_dollars,discount_dollars,original_order_id,original_line_id
164099145,3,bikna,2025-03-27T17:05:33Z,Sushant,123456,20001412,malh,2,5.95,0.0,null,null
